<a href="https://colab.research.google.com/github/dev-jotape/favela_environmental_trajectories/blob/main/scripts/01_download_mapbiomas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import ee
ee.Authenticate(auth_mode='notebook')   # open the link, log in, paste the code

# Save the credentials to Drive so the R runtime can use them
import os, shutil
creds_src = os.path.expanduser('~/.config/earthengine/credentials')
creds_dst = '/content/drive/MyDrive/Yale_Environmental_Data_Science_2026-2027/.earthengine_credentials'
shutil.copy(creds_src, creds_dst)
print('Credentials saved to', creds_dst)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Credentials saved to /content/drive/MyDrive/Yale_Environmental_Data_Science_2026-2027/.earthengine_credentials


### 0. Project folder

In [ ]:
BASE_DIR        <- "/content/drive/MyDrive/Yale_Environmental_Data_Science_2026-2027"
my_project_name <- "favela_environmental_trajectories"
project_path    <- file.path(BASE_DIR, my_project_name)

setwd(project_path)
getwd()   # should end in /favela_environmental_trajectories

[1] "/content/drive/MyDrive/Yale_Environmental_Data_Science_2026-2027/favela_environmental_trajectories"

### 1. Packages (fast setup for Colab)

In [ ]:
ubuntu  <- system("lsb_release -cs", intern = TRUE)
r_ver   <- paste(R.version$major, sub("\\..*", "", R.version$minor), sep = ".")
lib_dir <- file.path(BASE_DIR, paste0("R_libs_", r_ver, "_", ubuntu))
dir.create(lib_dir, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib_dir, .libPaths()))

# System libraries that sf links against (only installed if missing)
if (system("ldconfig -p | grep -q libproj", ignore.stdout = TRUE) != 0) {
  system("apt-get -qq update && apt-get -qq install -y libgdal-dev libgeos-dev libproj-dev libudunits2-dev > /dev/null")
}

options(
  repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", ubuntu)),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                          paste(getRversion(), R.version$platform,
                                R.version$arch, R.version$os)),
  Ncpus = parallel::detectCores()
)

needed  <- c("sf", "rgee")
missing <- needed[!vapply(needed, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing) > 0) install.packages(missing, lib = lib_dir)

# Use Colab's system Python (already has earthengine-api); skips ee_install()
if (system("python3 -c 'import ee'", ignore.stderr = TRUE) != 0) {
  system("pip install -q earthengine-api")
}
Sys.setenv(RETICULATE_PYTHON  = "/usr/bin/python3",
           EARTHENGINE_PYTHON = "/usr/bin/python3")

library(sf)
library(dplyr)
library(rgee)

also installing the dependencies ‘sp’, ‘terra’, ‘geometries’, ‘jsonify’, ‘rapidjsonr’, ‘sfheaders’, ‘RcppTOML’, ‘here’, ‘png’, ‘crosstalk’, ‘leaflet.providers’, ‘raster’, ‘geojsonsf’, ‘reticulate’, ‘leaflet’, ‘leafem’


Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.4.0; sf_use_s2() is TRUE


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




### 2. Settings

In [ ]:
GEE_PROJECT  <- "daytime-images"   # Earth Engine cloud project

SHAPEFILE    <- "data/raw/state_of_brazilian_favelas.shp"
RAW_DIR      <- "data/raw"

# Check column names first and set the two below:
#   names(st_read(SHAPEFILE, quiet = TRUE))
ID_COL       <- "CD_FCU"                # unique favela ID column (adjust)
MUN_COL      <- "CD_MUN"                # municipality code column (adjust)

PILOT        <- TRUE                    # TRUE = one city only
PILOT_MUN    <- "3550308"               # São Paulo (IBGE code; drop quotes if numeric)

# Full national run: upload the shapefile once as a GEE asset
# (Code Editor > Assets > New > Shape files) and set its path here.
FAVELA_ASSET <- "projects/daytime-images/assets/favelas_2022"

MAPBIOMAS    <- "projects/mapbiomas-public/assets/brazil/lulc/collection11/mapbiomas_brazil_collection11_coverage_v3"
YEARS        <- 1985:2025
SCALE        <- 30

# GEE exports to a folder NAME at the root of My Drive (no nested paths)
DRIVE_FOLDER <- "favela_environmental_trajectories_gee"
DRIVE_ROOT   <- "/content/drive/MyDrive"

### 3. Connect to Earth Engine

In [ ]:
CREDS_DRIVE <- file.path(BASE_DIR, ".earthengine_credentials")
CREDS_LOCAL <- path.expand("~/.config/earthengine/credentials")

if (!file.exists(CREDS_DRIVE)) {
  stop("Earth Engine credentials not found at ", CREDS_DRIVE, ".\n",
       "Colab's R runtime cannot run the interactive login. Generate the file ",
       "once with Step A (Python runtime) and run this again.")
}
dir.create(dirname(CREDS_LOCAL), recursive = TRUE, showWarnings = FALSE)
file.copy(CREDS_DRIVE, CREDS_LOCAL, overwrite = TRUE)

ee_Initialize(project = GEE_PROJECT, drive = FALSE)

[1] TRUE

── rgee 1.1.8 ──────────────────────────────────────── earthengine-api 1.7.39 ── 
 ✔ user: not_defined 
 ✔ Initializing Google Earth Engine:  DONE!
 ✔ Earth Engine account: projects/daytime-images/assets/favelas_2022 
 ✔ Python Path: /usr/bin/python3 
──────────────────────────────────────────────────────────────────────────────── 


### 4. Favela polygons

In [ ]:
favelas_all <- st_read("data/raw/state_of_brazilian_favelas.shp", quiet = TRUE)
names(favelas_all)
head(st_drop_geometry(favelas_all), 3)

[1] "CD_FCU"     "NM_FCU"     "CD_UF"      "NM_UF"      "SIGLA_UF"  
 [6] "CD_MUN"     "NM_MUN"     "distance"   "dist_km"    "dist_norm" 
[11] "area"       "hh_density" "age_idx"    "median_age" "sex_ratio" 
[16] "literate"   "race_wht"   "race_blk"   "race_ylw"   "race_brn"  
[21] "race_ind"   "race_unk"   "depriv_idx" "priv_hh"    "coll_hh"   
[26] "hh_type_1"  "hh_type_2"  "hh_type_3"  "hh_type_4"  "hh_type_5" 
[31] "hh_type_6"  "sanit_1"    "sanit_2"    "sanit_3"    "sanit_4"   
[36] "sanit_5"    "sanit_6"    "sanit_7"    "no_bath"    "excl_bath" 
[41] "shrd_bath"  "toilet"     "no_toilet"  "wtr_pipe"   "wtr_semi"  
[46] "no_wtr_pip" "wtr_dist"   "wtr_deep"   "wtr_shal"   "wtr_spring"
[51] "wtr_truck"  "wtr_rain"   "wtr_river"  "wtr_oth"    "wtr_area"  
[56] "veg_area"   "risk"       "slope_10"   "slope_20"   "slope_30"  
[61] "slope_30p"  "hand_6p"    "hand_3_6"   "hand_0_3"   "SO2"       
[66] "CO"         "O3"         "NO2"        "waste_coll" "demo_dens" 
[71] "est_dens"   "est_edu"    "est_health" "est_relig"  "est_agri"  
[76] "est_other"  "est_const"  "intra_veg"  "prim_hosp"  "inp_hosp"  
[81] "wtr_idx"    "sanit_idx"  "liv_a_idx"  "hh_idx"     "cluster"   
[86] "population" "area_km2"   "density_km" "geometry"

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,distance,dist_km,dist_norm,⋯,prim_hosp,inp_hosp,wtr_idx,sanit_idx,liv_a_idx,hh_idx,cluster,population,area_km2,density_km
,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,23646.847,23.646847,0.53315855,⋯,752.1925,2071.724,0.9974501,0.4510280,0.6812075,0.3215357,1,1159,0.026,45062.21
2,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,2544.012,2.544012,0.08035357,⋯,281.6608,1167.193,1.0000000,0.9788084,0.8729724,0.9998000,0,340,0.117,2916.00
3,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,28454.701,28.454701,0.50765095,⋯,0.0000,1003.263,1.0000000,1.0000000,0.8536706,0.9998000,0,394,0.028,14189.51


In [26]:
if (PILOT) {
  favelas <- st_read(SHAPEFILE, quiet = TRUE) %>%
    filter(.data[[MUN_COL]] == PILOT_MUN) %>%
    mutate(across(all_of(ID_COL), function(x) format(x, scientific = FALSE, trim = TRUE))) %>%
    st_make_valid() %>%
    st_transform(4326) %>%
    select(all_of(ID_COL))

  message("Pilot favelas: ", nrow(favelas))

  # sf -> GeoJSON -> ee$FeatureCollection (no geojsonio needed)
  tmp <- tempfile(fileext = ".geojson")
  st_write(favelas, tmp, driver = "GeoJSON", quiet = TRUE,
           layer_options = "COORDINATE_PRECISION=6")   # ~10 cm, keeps payload small
  fc <- jsonlite::fromJSON(tmp, simplifyVector = FALSE) %>%
    ee$FeatureCollection()
  print(fc$size()$getInfo())   # should match the number of pilot favelas
} else {
  fc <- ee$FeatureCollection(FAVELA_ASSET)$select(ID_COL)
}

Pilot favelas: 1357



[1] 1357


### 5. MapBiomas: all years as bands + pixel coordinates

In [ ]:
lulc  <- ee$Image(MAPBIOMAS)
bands <- paste0("classification_", YEARS)

available <- unlist(lulc$bandNames()$getInfo())
stopifnot(all(bands %in% available))   # stops if band names differ

img <- lulc$select(bands)$addBands(ee$Image$pixelLonLat())

### 6. One row per pixel inside each favela, on the native MapBiomas grid

(pixels whose center falls inside the polygon; very small favelas
may return no pixels)

In [ ]:
pixels <- img$sampleRegions(
  collection = fc,
  properties = list(ID_COL),
  projection = lulc$projection(),
  scale      = SCALE,
  tileScale  = 4,
  geometries = FALSE
)

### 7. Export to Drive and copy into data/raw

In [ ]:
suffix   <- if (PILOT) "pilot" else "brazil"
out_name <- paste0("mapbiomas_c11_favela_pixels_", suffix)
src      <- file.path(DRIVE_ROOT, DRIVE_FOLDER, paste0(out_name, ".csv"))

if (file.exists(src)) {
  message("Export already on Drive, skipping: ", src)
} else {
  task <- ee_table_to_drive(
    collection  = pixels,
    description = out_name,
    folder      = DRIVE_FOLDER,
    fileFormat  = "CSV",
    selectors   = as.list(c(ID_COL, "longitude", "latitude", bands)),
    timePrefix  = FALSE
  )
  task$start()
  ee_monitoring(task, max_attempts = 1000)
}

Polling for task <id: UF447JHRY7XPIUEX33CZLCLL, time: 0s>.
Polling for task <id: UF447JHRY7XPIUEX33CZLCLL, time: 5s>.
Polling for task <id: UF447JHRY7XPIUEX33CZLCLL, time: 10s>.
Polling for task <id: UF447JHRY7XPIUEX33CZLCLL, time: 15s>.
State: COMPLETED


In [ ]:
src

[1] "/content/drive/MyDrive/favela_environmental_trajectories_gee/mapbiomas_c11_favela_pixels_pilot.csv"

In [ ]:
# dir.create(RAW_DIR, recursive = TRUE, showWarnings = FALSE)
dest <- file.path(RAW_DIR, paste0(out_name, ".csv"))

if (file.exists(src)) {
  file.copy(src, dest, overwrite = TRUE)
  message("Copied to: ", dest, " (", round(file.size(dest) / 1e6, 1), " MB)")
} else {
  message("Not visible on the Drive mount yet. Wait a few minutes and run this cell again.")
}

Copied to: data/raw/mapbiomas_c11_favela_pixels_pilot.csv (13.2 MB)



### 8. Metadata (provenance for data stewardship)

In [ ]:
writeLines(c(
  paste("file:",            basename(dest)),
  paste("downloaded:",      format(Sys.time(), "%Y-%m-%d %H:%M %Z")),
  paste("source:",          "MapBiomas Brazil, Collection 11, land cover and land use"),
  paste("gee_asset:",       MAPBIOMAS),
  paste("years:",           paste(range(YEARS), collapse = "-")),
  paste("resolution_m:",    SCALE),
  paste("polygons:",        if (PILOT) paste("IBGE favelas 2022, municipality", PILOT_MUN)
                            else paste("IBGE favelas 2022, asset", FAVELA_ASSET)),
  "sampling: one row per 30 m pixel whose center falls inside a favela polygon",
  "license: MapBiomas data CC-BY-4.0; cite Souza et al. (2020), doi:10.3390/rs12172735",
  "legend: https://brasil.mapbiomas.org/codigos-de-legenda"
), file.path(RAW_DIR, paste0(out_name, "_metadata.txt")))

### 9. Quick check (no processing)

In [ ]:
raw <- read.csv(dest, nrows = 5)
str(raw)
message("Rows (pixels): ", length(readLines(dest)) - 1)

'data.frame':	5 obs. of  44 variables:
 $ CD_FCU             : num  3.55e+10 3.55e+10 3.55e+10 3.55e+10 3.55e+10
 $ longitude          : num  -46.4 -46.4 -46.4 -46.4 -46.4
 $ latitude           : num  -23.5 -23.5 -23.5 -23.5 -23.5
 $ classification_1985: int  24 24 24 24 24
 $ classification_1986: int  24 24 24 24 24
 $ classification_1987: int  24 24 24 24 24
 $ classification_1988: int  24 24 24 24 24
 $ classification_1989: int  24 24 24 24 24
 $ classification_1990: int  24 24 24 24 24
 $ classification_1991: int  24 24 24 24 24
 $ classification_1992: int  24 24 24 24 24
 $ classification_1993: int  24 24 24 24 24
 $ classification_1994: int  24 24 24 24 24
 $ classification_1995: int  24 24 24 24 24
 $ classification_1996: int  24 24 24 24 24
 $ classification_1997: int  24 24 24 24 24
 $ classification_1998: int  24 24 24 24 24
 $ classification_1999: int  24 24 24 24 24
 $ classification_2000: int  24 24 24 24 24
 $ classification_2001: int  24 24 24 24 24
 $ classification_200

Rows (pixels): 77264

